# MANUFACTURING QUALITY PERFORMANCE ANALYSIS
### 1. Data Loading and Initial Analysis

Objective:
Load the raw manufacturing quality dataset and check its 
structure, data types, missing values, duplicates, and data quality.

In [1]:
import numpy as np
import pandas as pd

#### Load Raw Dataset

In [2]:
# Load raw dataset
df = pd.read_csv("../data/product_quality_control_raw.csv", keep_default_na=False)
print(f"Dataset Shape: {df.shape}")
df.head(2)

Dataset Shape: (18313, 29)


,inspection_id,inspection_date,inspection_time,plant_location,production_line,machine_id,shift,inspection_type,inspector_id,product_id,...,tolerance_mm,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,inspection_result,root_cause_category,rework_cost_inr,scrap_cost_inr,downstream_complaint
0,QC-000001,2025-10-01,00:27:27,Noida,NOI-L3,NOI-L3-M3,Night,FQC,NaN,APP-3002,...,0.50,520.254,,190.1,48.0,Fail,Environment,0.0,5200.0,N
1,QC-000002,2025-10-01,01:06:17,Noida,NOI-L2,NOI-L2-M1,Night,FQC,INS-1024,APP-3004,...,0.25,180.161,431.9,188.7,47.1,Pass,,0.0,1560.0,N


In [3]:
# Summary of the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18313 entries, 0 to 18312
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   inspection_id            18313 non-null  str    
 1   inspection_date          18313 non-null  str    
 2   inspection_time          18313 non-null  str    
 3   plant_location           18313 non-null  str    
 4   production_line          18313 non-null  str    
 5   machine_id               18313 non-null  str    
 6   shift                    18313 non-null  str    
 7   inspection_type          18313 non-null  str    
 8   inspector_id             18313 non-null  str    
 9   product_id               18313 non-null  str    
 10  product_name             18313 non-null  str    
 11  product_category         18313 non-null  str    
 12  supplier_name            18313 non-null  str    
 13  batch_id                 18313 non-null  str    
 14  sample_size              18313 no

In [4]:
# Check missing or null values
df.isna().sum()

inspection_id              0
inspection_date            0
inspection_time            0
plant_location             0
production_line            0
machine_id                 0
shift                      0
inspection_type            0
inspector_id               0
product_id                 0
product_name               0
product_category           0
supplier_name              0
batch_id                   0
sample_size                0
units_defective            0
defect_type                0
defect_severity            0
spec_dimension_mm          0
tolerance_mm               0
measured_dimension_mm      0
measured_weight_g          0
process_temp_c             0
inspection_duration_min    0
inspection_result          0
root_cause_category        0
rework_cost_inr            0
scrap_cost_inr             0
downstream_complaint       0
dtype: int64

In [5]:
# Check common text placeholders in the dataset

placeholders = ["N/A", "NA", "NAN", "NULL", "NONE", "UNKNOWN", "-", "?", "NOT RECORDED", ""]

for col in df.columns:
    values = df[col].astype("string").str.strip().str.upper()
    count = values.isin(placeholders).sum()
    if count > 0:
        print(f"{col}: {count}")

inspection_time: 217
machine_id: 271
shift: 146
inspector_id: 645
supplier_name: 457
batch_id: 182
units_defective: 36
defect_type: 6903
defect_severity: 7441
measured_dimension_mm: 401
measured_weight_g: 330
process_temp_c: 730
inspection_duration_min: 363
root_cause_category: 10842
downstream_complaint: 182


#### Check Duplicate Records

In [6]:
# Check duplicate data
print("Duplicate rows:", df.duplicated().sum())
print(f"\nUnique inspection IDs: {df["inspection_id"].nunique()}")
print(f"Repeated inspection IDs: {df["inspection_id"].duplicated().sum()}")

duplicate_ids = df["inspection_id"].value_counts()

print(f"Inspection IDs appearing more than once: {(duplicate_ids > 1).sum()}")

Duplicate rows: 288

Unique inspection IDs: 18000
Repeated inspection IDs: 313
Inspection IDs appearing more than once: 312


In [7]:
columns_to_check = [
    "sample_size",
    "units_defective",
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min",
    "rework_cost_inr",
    "scrap_cost_inr"
]

for col in columns_to_check:
    print("\n", df[col].value_counts(dropna=False).head(10))


 sample_size
50        5405
32        4640
80        3529
20        2839
125       1836
50 pcs      22
32 pcs      15
80 pcs      11
20 pcs       7
0            7
Name: count, dtype: int64

 units_defective
0    7295
1    5240
2    2719
3    1386
4     728
5     378
6     214
7     110
8      56
9      34
Name: count, dtype: int64

 measured_dimension_mm
          164
NA         65
12.503     61
12.499     55
12.501     54
12.497     53
12.5       52
12.502     52
12.495     51
12.494     48
Name: count, dtype: int64

 measured_weight_g
        127
45.2    120
45.0    113
45.1    104
44.9     91
44.8     90
57.9     83
45.3     81
58.0     80
57.7     73
Name: count, dtype: int64

 process_temp_c
         271
N/A      102
NaN       95
null      93
-         89
NA        80
220.0     64
219.2     57
245.3     55
220.9     53
Name: count, dtype: int64

 inspection_duration_min
        139
22.1    127
22.2    126
16.1    119
22.0    117
22.7    114
21.2    114
21.3    113
15.8    113
16.

In [8]:
# Checking duplicate rows for each repeated ids
repeated_ids = df.loc[
    df["inspection_id"].duplicated(keep=False)
].sort_values("inspection_id")

display(repeated_ids.head(10))

,inspection_id,inspection_date,inspection_time,plant_location,production_line,machine_id,shift,inspection_type,inspector_id,product_id,...,tolerance_mm,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,inspection_result,root_cause_category,rework_cost_inr,scrap_cost_inr,downstream_complaint
65,QC-000066,2025-10-02,08:36:24,Aurangabad,AUR-L4,AUR-L4-M1,Morning,FQC,INS-1015,PKG-5002,...,1.00,450.42,366.2,165.6,23.8,fail,Method,0.0,28.0,N
66,QC-000066,2025-10-02,08:36:24,Aurangabad,AUR-L4,AUR-L4-M1,Morning,FQC,INS-1015,PKG-5002,...,1.00,450.42,366.2,165.6,25.8,Pass,Method,0.0,28.0,N
70,QC-000070,2025-10-02,09:09:16,Hosur,HSR-L4,HSR-L4-M3,Morning,IQC,INS-1029,APP-3002,...,0.50,519.991,3905.0,191.4,19.6,Pass,,0.0,0.0,N
8531,QC-000070,2025-10-02,09:09:16,Hosur,HSR-L4,HSR-L4-M3,Morning,IQC,INS-1029,APP-3002,...,0.50,519.991,3905.0,191.4,19.6,Pass,,0.0,0.0,N
89,QC-000089,2025-10-02,19:57:25,Noida,NOI-L2,NOI-L2-M1,Afternoon,IPQC,INS-1023,PKG-5001,...,0.40,209.878,48.6,159.3,32.3,Rework,Material,9.0,0.0,N
90,QC-000089,2025-10-02,19:57:25,Noida,NOI-L2,NOI-L2-M1,Afternoon,IPQC,INS-1023,PKG-5001,...,0.40,209.878,48.6,159.3,32.3,Rework,Material,9.0,0.0,N
103,QC-000102,2025-10-03,01:01:43,Pune,PUN-L4,PUN-L4-M1,Night,IPQC,INS-1003,AUT-1004,...,0.04,71.969,968.8,219.7,21.9,pass,,0.0,1450.0,N
104,QC-000102,2025-10-03,01:01:43,Pune,PUN-L4,PUN-L4-M1,Night,IPQC,INS-1003,AUT-1004,...,0.04,71.969,968.8,219.7,21.9,pass,,0.0,1450.0,N
241,QC-000236,2025-10-06,11:00:03,Chennai,CHN-L3,CHN-L3-M1,Morning,IPQC,INS-1014,PKG-5001,...,0.40,210.022,47.2,171.8,18.7,P,,0.0,Rs. 12,N
240,QC-000236,2025-10-06,11:00:03,Chennai,CHN-L3,CHN-L3-M1,Morning,IPQC,INS-1014,PKG-5001,...,0.40,210.022,47.2,171.8,18.7,P,,0.0,Rs. 12,N


In [9]:
# Get all rows belonging to repeated inspection IDs

# Keep only inspection IDs that occur more than once
repeated = df[df["inspection_id"].duplicated(keep=False)].copy()

# Count distinct full records per inspection ID
id_check = (
    repeated.drop_duplicates()
    .groupby("inspection_id")
    .size()
)

print("IDs with identical records:", (id_check == 1).sum())
print("IDs with conflicting records:", (id_check > 1).sum())


IDs with identical records: 287
IDs with conflicting records: 25


In [10]:
# Identify inspection IDs with conflicting records
conflicting_ids = id_check[id_check > 1].index

# Display all records for those IDs
conflicting_records = repeated[
    repeated["inspection_id"].isin(conflicting_ids)
].sort_values("inspection_id")

display(conflicting_records.head(10))

,inspection_id,inspection_date,inspection_time,plant_location,production_line,machine_id,shift,inspection_type,inspector_id,product_id,...,tolerance_mm,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,inspection_result,root_cause_category,rework_cost_inr,scrap_cost_inr,downstream_complaint
65,QC-000066,2025-10-02,08:36:24,Aurangabad,AUR-L4,AUR-L4-M1,Morning,FQC,INS-1015,PKG-5002,...,1.00,450.42,366.2,165.6,23.8,fail,Method,0.0,28.0,N
66,QC-000066,2025-10-02,08:36:24,Aurangabad,AUR-L4,AUR-L4-M1,Morning,FQC,INS-1015,PKG-5002,...,1.00,450.42,366.2,165.6,25.8,Pass,Method,0.0,28.0,N
647,QC-000637,14-Oct-2025,16:53:03,Aurangabad,AUR-L1,AUR-L1-M3,Afternoon,OQC,INS-1017,APP-3004,...,0.25,180.05,430.3,200.7,20.6,Pass,,0.0,520.0,N
648,QC-000637,14-Oct-2025,16:53:03,Aurangabad,AUR-L1,AUR-L1-M3,Afternoon,OQC,INS-1017,APP-3004,...,0.25,180.05,430.3,200.7,22.6,Rework,,0.0,520.0,N
1636,QC-001611,2025-11-04,12:31:53,Pune,PUN-L3,PUN-L3-M3,Morning,IQC,INS-1001,IND-4003,...,0.10,149.971,2306.5,260.7,18.1,Rework,Material,312.5,0.0,N
1637,QC-001611,2025-11-04,12:31:53,Pune,PUN-L3,PUN-L3-M3,Morning,IQC,INS-1001,IND-4003,...,0.10,149.971,2306.5,260.7,20.1,Pass,Material,312.5,0.0,N
1993,QC-001961,2025-11-11,06:58:37,Hosur,HSR-L1,HSR-L1-M2,Morning,IQC,INS-1033,ELE-2001,...,0.10,120.077,93.5,248.4,24.7,Pass,,0.0,720.0,N
1994,QC-001961,2025-11-11,06:58:37,Hosur,HSR-L1,HSR-L1-M2,Morning,IQC,INS-1033,ELE-2001,...,0.10,120.077,93.5,248.4,26.7,Rework,,0.0,720.0,N
2293,QC-002254,17/11/2025,07:15:54,Pune,pun-l2,PUN-L2-M1,Morning,IQC,INS-1003,IND-4004,...,0.15,62.948,378.0,253.1,21.8,Pass,,0.0,690.0,N
2294,QC-002254,17/11/2025,07:15:54,Pune,pun-l2,PUN-L2-M1,Morning,IQC,INS-1003,IND-4004,...,0.15,62.948,378.0,253.1,23.8,Rework,,0.0,690.0,N


#### Create Working Copy of Dataset

In [11]:
# Create a working copy of the raw dataset
df_work = df.copy()

# Count exact duplicate rows before removal
exact_duplicates = df_work.duplicated().sum()

# Remove exact duplicate rows
df_work = df_work.drop_duplicates().copy()

print("Exact duplicate rows removed:", exact_duplicates)
print("Rows before:", len(df))
print("Rows after:", len(df_work))
print("Remaining exact duplicates:", df_work.duplicated().sum())

Exact duplicate rows removed: 288
Rows before: 18313
Rows after: 18025
Remaining exact duplicates: 0


In [12]:
# Identify IDs associated with more than one distinct record
id_record_counts = (
    df_work.groupby("inspection_id")
    .apply(lambda group: len(group.drop_duplicates()))
)

conflicting_ids = id_record_counts[
    id_record_counts > 1
].index

# Flag all rows belonging to those IDs
df_work["duplicate_id_conflict"] = (
    df_work["inspection_id"].isin(conflicting_ids)
)

print("Conflicting inspection IDs:", len(conflicting_ids))
print(
    "Rows flagged for conflicting IDs:",
    df_work["duplicate_id_conflict"].sum()
)

Conflicting inspection IDs: 25
Rows flagged for conflicting IDs: 50


#### Standardizing Text Placeholders

In [13]:
# Standardize text placeholders as missing values
# Replaces specific placeholder values with NaN

placeholders = ["N/A", "NA", "NAN", "NULL", "UNKNOWN", "-", "?", "NOT RECORDED", ""]

for col in df_work.select_dtypes(include="str").columns:
    df_work[col] = df_work[col].str.strip()
    df_work.loc[df_work[col].str.upper().isin(placeholders), col] = np.nan

# Check missing values after standardization
missing_values = df_work.isna().sum()

display(missing_values[missing_values > 0].sort_values(ascending=False))

root_cause_category        10670
process_temp_c               720
inspector_id                 630
supplier_name                451
measured_dimension_mm        396
inspection_duration_min      360
measured_weight_g            324
machine_id                   270
defect_type                  270
inspection_time              216
batch_id                     181
defect_severity              180
downstream_complaint         180
shift                        144
units_defective               36
dtype: int64

In [14]:
df_work.head(2)

,inspection_id,inspection_date,inspection_time,plant_location,production_line,machine_id,shift,inspection_type,inspector_id,product_id,...,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,inspection_result,root_cause_category,rework_cost_inr,scrap_cost_inr,downstream_complaint,duplicate_id_conflict
0,QC-000001,2025-10-01,00:27:27,Noida,NOI-L3,NOI-L3-M3,Night,FQC,NaN,APP-3002,...,520.254,NaN,190.1,48.0,Fail,Environment,0.0,5200.0,N,False
1,QC-000002,2025-10-01,01:06:17,Noida,NOI-L2,NOI-L2-M1,Night,FQC,INS-1024,APP-3004,...,180.161,431.9,188.7,47.1,Pass,NaN,0.0,1560.0,N,False


#### Columns that should contain only Numeric Values

In [15]:
# Columns that should contain numbers

numeric_columns = [
    "sample_size",
    "units_defective",
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min",
    "rework_cost_inr",
    "scrap_cost_inr"
]

# Check for values that are not numbers
for col in numeric_columns:
    numbers = pd.to_numeric(df_work[col], errors="coerce")
    
    invalid_values = df_work[col][df_work[col].notna() & numbers.isna()]
    
    print("\nColumn:", col)
    print("Invalid values:", len(invalid_values))
    
    if len(invalid_values) > 0:
        display(invalid_values.value_counts().head(10))



Column: sample_size
Invalid values: 55


sample_size
50 pcs     21
32 pcs     15
80 pcs     11
20 pcs      6
125 pcs     2
Name: count, dtype: int64


Column: units_defective
Invalid values: 0

Column: measured_dimension_mm
Invalid values: 0

Column: measured_weight_g
Invalid values: 0

Column: process_temp_c
Invalid values: 0

Column: inspection_duration_min
Invalid values: 0

Column: rework_cost_inr
Invalid values: 829


rework_cost_inr
₹0.00      370
Rs. 0      251
TBD         54
Rs. 7        9
₹172.50      4
Rs. 180      4
Rs. 3        4
₹650.00      3
₹462.50      3
₹115.00      3
Name: count, dtype: int64


Column: scrap_cost_inr
Invalid values: 873


scrap_cost_inr
₹0.00        257
Rs. 0        211
TBD           54
Rs. 12         7
₹2,200.00      7
₹5,200.00      7
3,400.00       7
₹12.00         7
Rs. 2,200      7
2,600.00       6
Name: count, dtype: int64

#### Converting Non-Numeric Values to Numeric Values

In [16]:
# Clean sample_size
df_work["sample_size"] = df_work["sample_size"].astype("string")
df_work["sample_size"] = df_work["sample_size"].str.replace(
    "pcs", "", case=False
).str.strip()

df_work["sample_size"] = pd.to_numeric(df_work["sample_size"], errors="coerce")

# Convert units_defective to numbers
df_work["units_defective"] = pd.to_numeric(df_work["units_defective"], errors="coerce")

# Clean cost columns
for col in ["rework_cost_inr", "scrap_cost_inr"]:
    df_work[col] = df_work[col].astype("string")

    # Remove currency symbols, Rs, and commas
    df_work[col] = df_work[col].str.replace(r"₹|Rs\.?", "", regex=True, case=False)
    df_work[col] = df_work[col].str.replace(",", "", regex=False)

    # Convert to numbers
    df_work[col] = pd.to_numeric(df_work[col].str.strip(), errors="coerce")

# Convert measurement columns to numbers
measurement_columns = [
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min"
]

for col in measurement_columns:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

# Check results
print(df_work[numeric_columns].dtypes)
display(df_work[numeric_columns].isna().sum())

sample_size                  Int64
units_defective            float64
measured_dimension_mm      float64
measured_weight_g          float64
process_temp_c             float64
inspection_duration_min    float64
rework_cost_inr            Float64
scrap_cost_inr             Float64
dtype: object


sample_size                  0
units_defective             36
measured_dimension_mm      396
measured_weight_g          324
process_temp_c             720
inspection_duration_min    360
rework_cost_inr             54
scrap_cost_inr              54
dtype: int64

#### Count Missing Values

In [17]:
# Count missing values in each column
display(df_work.isna().sum())

inspection_id                  0
inspection_date                0
inspection_time              216
plant_location                 0
production_line                0
machine_id                   270
shift                        144
inspection_type                0
inspector_id                 630
product_id                     0
product_name                   0
product_category               0
supplier_name                451
batch_id                     181
sample_size                    0
units_defective               36
defect_type                  270
defect_severity              180
spec_dimension_mm              0
tolerance_mm                   0
measured_dimension_mm        396
measured_weight_g            324
process_temp_c               720
inspection_duration_min      360
inspection_result              0
root_cause_category        10670
rework_cost_inr               54
scrap_cost_inr                54
downstream_complaint         180
duplicate_id_conflict          0
dtype: int

In [18]:
# Check data types of all columns
print(df_work.dtypes)

inspection_id                  str
inspection_date                str
inspection_time                str
plant_location                 str
production_line                str
machine_id                     str
shift                          str
inspection_type                str
inspector_id                   str
product_id                     str
product_name                   str
product_category               str
supplier_name                  str
batch_id                       str
sample_size                  Int64
units_defective            float64
defect_type                    str
defect_severity                str
spec_dimension_mm          float64
tolerance_mm               float64
measured_dimension_mm      float64
measured_weight_g          float64
process_temp_c             float64
inspection_duration_min    float64
inspection_result              str
root_cause_category            str
rework_cost_inr            Float64
scrap_cost_inr             Float64
downstream_complaint

#### Summary of Numeric Columns

In [19]:
# Summary of numeric columns
display(df_work[numeric_columns].describe())

,sample_size,units_defective,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,rework_cost_inr,scrap_cost_inr
count,18025.0,17989.000000,17629.000000,17701.000000,17305.000000,17665.000000,17971.0,17971.0
mean,54.067184,1.412808,152.340831,1268.708638,233.231615,24.482145,99.865366,908.998275
std,30.741013,3.607050,149.117662,1778.087212,383.085896,45.917386,362.431923,2196.646629
min,0.0,-3.000000,1.250000,-999.000000,-999.000000,-35.200000,0.0,0.0
25%,32.0,0.000000,64.940000,119.700000,192.500000,15.700000,0.0,0.0
50%,50.0,1.000000,92.080000,379.600000,223.000000,21.100000,0.0,0.0
75%,80.0,2.000000,209.914000,1435.400000,247.300000,30.100000,0.0,920.0
max,125.0,130.000000,4505.700000,6350.100000,9999.000000,2742.000000,7800.0,36400.0


#### Check Invalid Counts & Cost Values

In [20]:
# Check invalid counts
print("Zero sample sizes:",
      (df_work["sample_size"] == 0).sum())

print("Negative sample sizes:",
      (df_work["sample_size"] < 0).sum())

print("Negative defective counts:",
      (df_work["units_defective"] < 0).sum())

print("Defective units greater than sample size:",
      (df_work["units_defective"] > df_work["sample_size"]).sum())

# Check suspicious measurement and cost values
for col in [
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min",
    "rework_cost_inr",
    "scrap_cost_inr"
]:
    print(
        f"{col}:",
        "Negative =", (df_work[col] < 0).sum(),
        "| Zero =", (df_work[col] == 0).sum()
    )

Zero sample sizes: 6
Negative sample sizes: 0
Negative defective counts: 8
Defective units greater than sample size: 49
measured_dimension_mm: Negative = 0 | Zero = 0
measured_weight_g: Negative = 33 | Zero = 38
process_temp_c: Negative = 25 | Zero = 0
inspection_duration_min: Negative = 10 | Zero = 0
rework_cost_inr: Negative = 0 | Zero = 14371
scrap_cost_inr: Negative = 0 | Zero = 10710


#### Check Invalid Sample Sizes and Defective Counts

In [21]:
# Check invalid sample sizes and defective counts
count_issues = df_work[
    (df_work["sample_size"] == 0)
    | (df_work["units_defective"] < 0)
    | (
        df_work["units_defective"]
        > df_work["sample_size"]
    )
]

display(
    count_issues[
        [
            "inspection_id",
            "sample_size",
            "units_defective",
            "inspection_result",
            "defect_type",
            "defect_severity"
        ]
    ].head(10)
)

print("Total records with count issues:", len(count_issues))

,inspection_id,sample_size,units_defective,inspection_result,defect_type,defect_severity
175,QC-000172,32,34.0,Pass,None,None
978,QC-000963,125,127.0,Pass,Dimensional Deviation,Major
2315,QC-002275,0,0.0,Pass,None,None
2559,QC-002511,20,21.0,Pass,None,None
2792,QC-002738,50,55.0,Passed,No Defect,None
3275,QC-003211,32,35.0,Pass,None,None
3895,QC-003822,32,35.0,Pass,None,None
3920,QC-003846,32,-3.0,Pass,None,None
4036,QC-003961,32,36.0,Pass,None,None
4064,QC-003989,50,51.0,Pass,None,None


Total records with count issues: 59


#### Check Invalid Measurements & Durations

In [22]:
# Inspect suspicious physical measurements and durations
measurement_issues = df_work[
    (df_work["measured_weight_g"] < 0)
    | (df_work["process_temp_c"] < 0)
    | (df_work["inspection_duration_min"] < 0)
]

display(
    measurement_issues[
        [
            "inspection_id",
            "product_name",
            "measured_weight_g",
            "process_temp_c",
            "inspection_duration_min"
        ]
    ].head(10)
)

print(
    "Total records with suspicious measurements:",
    len(measurement_issues)
)

,inspection_id,product_name,measured_weight_g,process_temp_c,inspection_duration_min
340,QC-000334,HDPE Bottle 1L,-999.0,157.4,45.1
579,QC-000570,Brake Disc Rotor,-999.0,224.0,21.7
680,QC-000669,LED Driver Module,-999.0,242.9,19.7
695,QC-000684,Fuel Injector Nozzle,45.3,-999.0,15.7
844,QC-000832,Fuel Injector Nozzle,-999.0,226.3,27.5
969,QC-000954,Wheel Hub Bearing,985.7,-999.0,21.3
1264,QC-001244,Ball Valve 2in,1258.3,-999.0,20.5
2230,QC-002193,Gear Shaft,764.6,-999.0,13.2
2329,QC-002289,Lithium Battery Pack,319.7,248.4,-20.7
2499,QC-002453,HDPE Bottle 1L,-999.0,NaN,17.4


Total records with suspicious measurements: 68


#### Fix Negative Inspection Durations

In [23]:
# Fix negative inspection durations
# converting negative values into postitive
print("Negative durations before:", (df_work["inspection_duration_min"] < 0).sum())

df_work["inspection_duration_min"] = df_work["inspection_duration_min"].abs()

print("Negative durations after:", (df_work["inspection_duration_min"] < 0).sum())

Negative durations before: 10
Negative durations after: 0


In [24]:
# 1. Mark invalid sample sizes as missing
df_work.loc[
    df_work["sample_size"] <= 0,
    "sample_size"
] = np.nan

# 2. Mark invalid defective counts as missing
invalid_defect_count = (
    (df_work["units_defective"] < 0)
    | (
        df_work["units_defective"]
        > df_work["sample_size"]
    )
)

df_work.loc[
    invalid_defect_count,
    "units_defective"
] = np.nan

# 3. Mark sentinel values as missing
for col in ["measured_weight_g", "process_temp_c"]:
    df_work.loc[df_work[col] == -999, col] = np.nan

# 4. Mark negative measurements and durations as missing
for col in [
    "measured_dimension_mm",
    "measured_weight_g",
    "inspection_duration_min"
]:
    df_work.loc[df_work[col] < 0, col] = np.nan

# 5. Verify the corrections
print("Sample sizes <= 0:",
      (df_work["sample_size"] <= 0).sum())

print("Negative defective counts:",
      (df_work["units_defective"] < 0).sum())

print("Defective counts greater than sample size:",
      (
          df_work["units_defective"]
          > df_work["sample_size"]
      ).sum())

print("Remaining -999 in weight:",
      (df_work["measured_weight_g"] == -999).sum())

print("Remaining -999 in temperature:",
      (df_work["process_temp_c"] == -999).sum())

print("Negative inspection durations:",
      (df_work["inspection_duration_min"] < 0).sum())

Sample sizes <= 0: 0
Negative defective counts: 0
Defective counts greater than sample size: 0
Remaining -999 in weight: 0
Remaining -999 in temperature: 0
Negative inspection durations: 0


In [25]:
# Check missing values in important numeric columns
numeric_columns = [
    "sample_size",
    "units_defective",
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min",
    "rework_cost_inr",
    "scrap_cost_inr"
]

print("Missing values after cleaning:")
print(df_work[numeric_columns].isna().sum())

print("\nRows with missing or invalid sample size:")
print(df_work[df_work["sample_size"].isna()][
    ["inspection_id", "sample_size", "units_defective"]
].head(10))

Missing values after cleaning:
sample_size                  6
units_defective             89
measured_dimension_mm      396
measured_weight_g          357
process_temp_c             745
inspection_duration_min    360
rework_cost_inr             54
scrap_cost_inr              54
dtype: int64

Rows with missing or invalid sample size:
      inspection_id  sample_size  units_defective
2315      QC-002275         <NA>              0.0
11598     QC-011402         <NA>              1.0
14715     QC-014460         <NA>              1.0
15441     QC-015172         <NA>              0.0
15552     QC-015282         <NA>              1.0
17962     QC-017651         <NA>              1.0


In [26]:
# Inspect the lowest and highest values in each column
columns_to_check = [
    "measured_dimension_mm",
    "measured_weight_g",
    "process_temp_c",
    "inspection_duration_min",
    "rework_cost_inr",
    "scrap_cost_inr"
]

for col in columns_to_check:
    print(f"\n--- {col} ---")

    print("Lowest values:")
    print(df_work[col].nsmallest(5).to_list())

    print("Highest values:")
    print(df_work[col].nlargest(5).to_list())


--- measured_dimension_mm ---
Lowest values:
[1.25, 5.079, 6.301, 6.498, 6.502]
Highest values:
[4505.7, 4502.59, 2801.54, 2800.55, 2800.17]

--- measured_weight_g ---
Lowest values:
[0.0, 0.0, 0.0, 0.0, 0.0]
Highest values:
[6350.1, 6322.1, 6315.9, 6315.3, 6308.9]

--- process_temp_c ---
Lowest values:
[144.9, 147.2, 148.1, 148.7, 149.0]
Highest values:
[9999.0, 9999.0, 9999.0, 9999.0, 9999.0]

--- inspection_duration_min ---
Lowest values:
[5.3, 6.1, 6.7, 6.7, 6.9]
Highest values:
[2742.0, 2712.0, 2574.0, 1818.0, 1326.0]

--- rework_cost_inr ---
Lowest values:
[0.0, 0.0, 0.0, 0.0, 0.0]
Highest values:
[7800.0, 6500.0, 6500.0, 6500.0, 6500.0]

--- scrap_cost_inr ---
Lowest values:
[0.0, 0.0, 0.0, 0.0, 0.0]
Highest values:
[36400.0, 34800.0, 31200.0, 31200.0, 26400.0]


#### Fix Measurements, Temparature & Dimension Values

In [27]:
# 1. Weight 0 and temperature 9999 are "no reading" codes, not real values
df_work.loc[df_work["measured_weight_g"] == 0, "measured_weight_g"] = np.nan
df_work.loc[df_work["process_temp_c"] == 9999, "process_temp_c"] = np.nan

# 2. Durations above 120 minutes are seconds, so convert them to minutes
in_seconds = df_work["inspection_duration_min"] > 120
df_work.loc[in_seconds, "inspection_duration_min"] = df_work.loc[in_seconds, "inspection_duration_min"] / 60

# 3. Dimensions 10x too big or 10x too small (compare with spec_dimension_mm)
ratio = df_work["measured_dimension_mm"] / df_work["spec_dimension_mm"]
too_big = ratio > 5
too_small = ratio < 0.2
df_work.loc[too_big, "measured_dimension_mm"] = df_work.loc[too_big, "measured_dimension_mm"] / 10
df_work.loc[too_small, "measured_dimension_mm"] = df_work.loc[too_small, "measured_dimension_mm"] * 10

# 4. Verify the corrections
print("Weight = 0:", (df_work["measured_weight_g"] == 0).sum())
print("Temperature = 9999:", (df_work["process_temp_c"] == 9999).sum())
print("Durations above 120:", (df_work["inspection_duration_min"] > 120).sum())
print("Dimension fixed:", too_big.sum() + too_small.sum())

Weight = 0: 0
Temperature = 9999: 0
Durations above 120: 0
Dimension fixed: 52


In [28]:
print((df_work["process_temp_c"] == 9999).sum())

0


#### Standarizing Text Formatting

In [29]:
# Standardize text formatting
for col in df_work.select_dtypes(include="str").columns:
    df_work[col] = df_work[col].str.strip()

# Standardize plant location
df_work["plant_location"] = df_work["plant_location"].str.title()

df_work["plant_location"] = df_work["plant_location"].str.replace(
    r"^(Aurangabad|Chhatrapati Sambhajinagar)$",
    "Chhatrapati Sambhaji Nagar",
    case=False,
    regex=True
)

# Replace AUR prefix with CSN in relevant ID columns
id_columns = [
    "production_line",
    "machine_id",
    "batch_id"
]

for col in id_columns:
    df_work[col] = df_work[col].str.upper().str.replace(
        r"^AUR", "CSN", case=False, regex=True
    )

In [30]:
df_work.head(5)

,inspection_id,inspection_date,inspection_time,plant_location,production_line,machine_id,shift,inspection_type,inspector_id,product_id,...,measured_dimension_mm,measured_weight_g,process_temp_c,inspection_duration_min,inspection_result,root_cause_category,rework_cost_inr,scrap_cost_inr,downstream_complaint,duplicate_id_conflict
0,QC-000001,2025-10-01,00:27:27,Noida,NOI-L3,NOI-L3-M3,Night,FQC,NaN,APP-3002,...,520.254,NaN,190.1,48.0,Fail,Environment,0.0,5200.0,N,False
1,QC-000002,2025-10-01,01:06:17,Noida,NOI-L2,NOI-L2-M1,Night,FQC,INS-1024,APP-3004,...,180.161,431.9,188.7,47.1,Pass,NaN,0.0,1560.0,N,False
2,QC-000003,2025-10-01,03:13:35,Chhatrapati Sambhaji Nagar,CSN-L3,CSN-L3-M3,Night,IPQC,INS-1020,PKG-5002,...,450.347,388.1,162.1,12.6,Rework,Machine,7.0,0.0,N,False
3,QC-000004,2025-10-01,03:50:30,Chhatrapati Sambhaji Nagar,CSN-L4,CSN-L4-M2,Night,IPQC,INS-1018,AUT-1004,...,72.003,990.2,215.2,43.0,Rework,Machine,1812.5,0.0,N,False
4,QC-000005,01-Oct-2025,04:21:16,Pune,PUN-L4,PUN-L4-M1,Night,IPQC,INS-1004,IND-4003,...,149.997,2306.3,267.6,24.4,Pass,NaN,0.0,0.0,N,False


In [31]:
# Inspector IDs: "ins-1005" and "INS 1005" -> "INS-1005"
df_work["inspector_id"] = df_work["inspector_id"].str.upper().str.replace(" ", "-", regex=False)

# Supplier names - keeps only the name and then add 'Pvt Ltd' at ending for all values
core_name = (
    df_work["supplier_name"].str.lower()
    .str.replace(".", "", regex=False)
    .str.replace(r"\bpvt\b|\bltd\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df_work["supplier_name"] = core_name.str.title() + " Pvt Ltd"

# Product name and category - group by product_id
# Same spelling for each product_id
for col in ["product_name", "product_category"]:
    spelling = df_work.groupby("product_id")[col].agg(lambda s: s.mode().iloc[0])
    df_work[col] = df_work["product_id"].map(spelling)

print("Suppliers:", df_work["supplier_name"].nunique())
print("Product categories:", df_work["product_category"].nunique())
print("Plants:", df_work["plant_location"].nunique())

Suppliers: 10
Product categories: 5
Plants: 5


In [32]:
print(df_work["inspector_id"].head(5))
print(df_work["supplier_name"].head(5))

0         NaN
1    INS-1024
2    INS-1020
3    INS-1018
4    INS-1004
Name: inspector_id, dtype: str
0    Bharat Precision Components Pvt Ltd
1                Orion Fasteners Pvt Ltd
2      Metro Packaging Solutions Pvt Ltd
3                Orion Fasteners Pvt Ltd
4                Orion Fasteners Pvt Ltd
Name: supplier_name, dtype: str


#### Fix Date Format

In [33]:
# Reload the original date values
df_work["inspection_date"] = df["inspection_date"].str.strip()
date_text = df_work["inspection_date"]

# Convert mixed date formats to one format
df_work["inspection_date"] = pd.to_datetime(df_work["inspection_date"], format="%Y-%m-%d", errors="coerce")

for date_format in ["%d/%m/%Y", "%d-%b-%Y", "%b %d, %Y"]:
    df_work["inspection_date"] = df_work["inspection_date"].fillna(pd.to_datetime(date_text, format=date_format, errors="coerce"))

print(f"Invalid or missing dates: {df_work["inspection_date"].isna().sum()}")

Invalid or missing dates: 0


In [34]:
print(df_work["inspection_date"].dtypes)
print(df_work["inspection_date"].head(5))

datetime64[us]
0   2025-10-01
1   2025-10-01
2   2025-10-01
3   2025-10-01
4   2025-10-01
Name: inspection_date, dtype: datetime64[us]


#### Standarized Capitalization & Remove Extra Spaces

In [35]:
# Normalize capitalization and spaces
df_work["inspection_result"] = (
    df_work["inspection_result"].str.strip().str.lower()
)

# Standardize equivalent values
df_work["inspection_result"] = df_work["inspection_result"].replace({
    "passed": "pass",
    "p": "pass",
    "f": "fail",
    "ok": "pass",
    "failed": "fail",
    "rejected": "fail",
    "re-work": "rework"
})

# Use consistent capitalization
df_work["inspection_result"] = df_work["inspection_result"].str.title()

print(df_work["inspection_result"].value_counts(dropna=False))

inspection_result
Pass      10728
Fail       3669
Rework     3628
Name: count, dtype: int64


In [36]:
print(df_work.iloc[30]["inspection_result"])
print(df_work.iloc[3]["inspection_result"])
print(df_work.iloc[5]["inspection_result"])

Pass
Rework
Fail


In [37]:
for col in [
    "shift",
    "inspection_type",
    "defect_severity",
    "root_cause_category",
    "downstream_complaint"
]:
    print(f"\n{col}:")
    print(df_work[col].value_counts().head(10))


shift:
shift
Morning      7083
Afternoon    6137
Night        4218
MORNING        92
morning        89
afternoon      74
AFTERNOON      70
night          65
NIGHT          53
Name: count, dtype: int64

inspection_type:
inspection_type
IPQC    6073
FQC     5308
IQC     3496
OQC     2608
ipqc     195
fqc      178
iqc      101
oqc       66
Name: count, dtype: int64

defect_severity:
defect_severity
None        6789
Minor       4540
Major       3866
Critical    1763
none         347
Min          269
Maj          183
Crit          88
Name: count, dtype: int64

root_cause_category:
root_cause_category
Material       1686
Machine        1585
Man            1554
Method         1222
Environment    1016
material         74
man              66
machine          64
method           54
environment      34
Name: count, dtype: int64

downstream_complaint:
downstream_complaint
N    17701
Y      144
Name: count, dtype: int64


In [38]:
# Standardize capitalization and remove extra spaces
for col in [
    "shift",
    "inspection_type",
    "defect_severity",
    "root_cause_category"
]:
    df_work[col] = df_work[col].str.strip().str.title()

# Standardize complaint values
df_work["downstream_complaint"] = (
    df_work["downstream_complaint"].str.strip().str.upper()
)

print(df_work["shift"].value_counts())
print(df_work["inspection_type"].value_counts())
print(df_work["downstream_complaint"].value_counts())
print(df_work["defect_severity"].value_counts())
print(df_work["root_cause_category"].value_counts())


shift
Morning      7264
Afternoon    6281
Night        4336
Name: count, dtype: int64
inspection_type
Ipqc    6268
Fqc     5486
Iqc     3597
Oqc     2674
Name: count, dtype: int64
downstream_complaint
N    17701
Y      144
Name: count, dtype: int64
defect_severity
None        7136
Minor       4540
Major       3866
Critical    1763
Min          269
Maj          183
Crit          88
Name: count, dtype: int64
root_cause_category
Material       1760
Machine        1649
Man            1620
Method         1276
Environment    1050
Name: count, dtype: int64


#### Fix Labels

In [39]:
# fix labels
df_work["inspection_type"] = df_work["inspection_type"].replace({
    "Ipqc": "IPQC",
    "Fqc": "FQC",
    "Iqc": "IQC",
    "Oqc": "OQC"
})

In [40]:
# fix labels
df_work["defect_severity"] = df_work["defect_severity"].replace({
    "Min": "Minor",
    "Maj": "Major",
    "Crit": "Critical"
})

print(df_work["defect_severity"].value_counts(dropna=False))

defect_severity
None        7136
Minor       4809
Major       4049
Critical    1851
NaN          180
Name: count, dtype: int64


#### Fix Time Format

In [41]:
# Converting inspection time in one format - (H:M:S)
time_text = df_work["inspection_time"]
time_parsed = pd.to_datetime(time_text, format="%H:%M:%S", errors="coerce")

for time_format in ["%H:%M", "%I:%M %p"]:
    time_parsed = time_parsed.fillna(pd.to_datetime(time_text, format=time_format, errors="coerce"))

df_work["inspection_time"] = time_parsed.dt.strftime("%H:%M:%S")

# Standarizing shift - Morning 06-13, Afternoon 14-21, Night 22-05
hour = time_parsed.dt.hour
shift_from_time = pd.Series(
    np.select([hour.between(6, 13), hour.between(14, 21)], ["Morning", "Afternoon"], default="Night"),
    index=df_work.index).where(hour.notna())

print("Blank shifts filled from time:", (df_work["shift"].isna() & shift_from_time.notna()).sum())
df_work["shift"] = df_work["shift"].fillna(shift_from_time)

print(df_work["inspection_time"].str.len().value_counts(dropna=False))

Blank shifts filled from time: 142
inspection_time
8.0    17809
NaN      216
Name: count, dtype: int64


In [42]:
# defect types
for col in ["defect_type"]:
    print(f"\n{col}:")
    print(df_work[col].value_counts(dropna=False))


defect_type:
defect_type
None                     6516
Dimensional Deviation    1669
Surface Scratch          1297
Wrong Label              1022
Leak                      929
Contamination             811
Functional Failure        663
Porosity                  656
Crack                     626
Burr                      621
Coating Defect            577
No Defect                 568
Solder Defect             442
Packaging Damage          320
Print Defect              292
NaN                       270
Dimension Error           142
Scratch                   117
functional failure         32
WRONG LABEL                32
wrong label                30
CONTAMINATION              29
leak                       26
contamination              24
LEAK                       23
POROSITY                   23
FUNCTIONAL FAILURE         20
Wrong_Label                18
Functional_Failure         18
BURR                       17
PACKAGING DAMAGE           16
COATING DEFECT             16
porosity      

#### Standarized Same Names for Similar Defect Types

In [43]:
# Standardizing same names for similar defect types

# replace any '_' with " "
df_work["defect_type"] = (df_work["defect_type"].str.lower().str.replace("_", " ", regex=False))

df_work["defect_type"] = df_work["defect_type"].replace({
    "scratch": "Surface Scratch",
    "dimension error": "Dimensional Deviation",
    "dimension deviation": "Dimensional Deviation",
    "no defect" : "None"
})

df_work["defect_type"] = df_work["defect_type"].str.title()

print(df_work["defect_type"].value_counts(dropna=False))

defect_type
None                     7084
Dimensional Deviation    1811
Surface Scratch          1414
Wrong Label              1102
Leak                      978
Contamination             864
Functional Failure        733
Porosity                  695
Crack                     654
Burr                      651
Coating Defect            623
Solder Defect             471
Packaging Damage          358
Print Defect              317
NaN                       270
Name: count, dtype: int64


In [44]:
# Standardizing units_defective by defect_type
# No defective units means defect "None", not unknown

# If the defect type is "None", units_defective must be 0
df_work.loc[(df_work["defect_type"] == "None") & df_work["units_defective"].isna(), "units_defective"] = 0

# If units_defective is 0, the defect type and severity are "None"
no_defect = df_work["units_defective"] == 0
df_work.loc[no_defect & df_work["defect_type"].isna(), "defect_type"] = "None"
df_work.loc[no_defect & df_work["defect_severity"].isna(), "defect_severity"] = "None"

print(f"Defect type still missing: {df_work["defect_type"].isna().sum()}")
print(f"Defect severity still missing: {df_work["defect_severity"].isna().sum()}")

Defect type still missing: 151
Defect severity still missing: 113


In [45]:
for col in ["rework_cost_inr", "scrap_cost_inr"]:
    print(col, "Negative values:", (df_work[col] < 0).sum())

print(df_work["scrap_cost_inr"].dtype)
print(df_work["rework_cost_inr"].dtype)

rework_cost_inr Negative values: 0
scrap_cost_inr Negative values: 0
Float64
Float64


In [46]:
print(df_work["duplicate_id_conflict"].value_counts())

duplicate_id_conflict
False    17975
True        50
Name: count, dtype: int64


In [47]:
print("Cleaned dataset shape:", df_work.shape)

print("\nExact duplicate rows:", df_work.duplicated().sum())
print("Conflicting rows:", df_work["duplicate_id_conflict"].sum())

print("\nMissing values:")
print(df_work.isna().sum()[df_work.isna().sum() > 0])

Cleaned dataset shape: (18025, 30)

Exact duplicate rows: 0
Conflicting rows: 50

Missing values:
inspection_time              216
machine_id                   270
shift                          2
inspector_id                 630
supplier_name                451
batch_id                     181
sample_size                    6
units_defective               56
defect_type                  151
defect_severity              113
measured_dimension_mm        396
measured_weight_g            395
process_temp_c               771
inspection_duration_min      360
root_cause_category        10670
rework_cost_inr               54
scrap_cost_inr                54
downstream_complaint         180
dtype: int64


#### Fix Wrong Dates

In [48]:
# Correct the wrong dates - (1970 & 2027)
wrong_dates = df_work['inspection_date'].dt.year.isin([1970, 2027])
print("Wrong dates repaired:", wrong_dates.sum())

df_work.loc[wrong_dates, 'inspection_date'] = pd.NaT
df_work = df_work.sort_values('inspection_id')
df_work['inspection_date'] = df_work['inspection_date'].ffill().bfill()

print("Records after date cleaning:", len(df_work))

Wrong dates repaired: 13
Records after date cleaning: 18025


#### Remove Duplicate Records

In [49]:
# Remove duplicate IDs and keeping the correct copy (not just the first one)
print("Duplicate IDs:", df_work['inspection_id'].duplicated().sum())

# Correct row has - (if inspection_result is Pass = no root cause)
df_work['row_is_consistent'] = (
    (df_work['inspection_result'] == 'Pass') == df_work['root_cause_category'].isna()
)

# Keeping consistent copy first and then keep one row per inspection ID
df_work = (
    df_work.sort_values(['inspection_id', 'row_is_consistent'], ascending=[True, False], kind='stable')
    .drop_duplicates(subset='inspection_id', keep='first')
    .drop(columns='row_is_consistent')
    .copy()
)

print("Final records:", len(df_work))

Duplicate IDs: 25
Final records: 18000


#### Fix "Pass" Label

In [50]:
# Fix "Pass" labels because root_cause_category only exists for Rework/Fail

wrong_pass = (df_work['inspection_result'] == 'Pass') & df_work['root_cause_category'].notna()
print("Wrong Pass labels corrected:", wrong_pass.sum())

# Rework cost above 0 means it was reworked, otherwise it failed
df_work.loc[wrong_pass, 'inspection_result'] = np.where(
    df_work.loc[wrong_pass, 'rework_cost_inr'] > 0, 'Rework', 'Fail'
)

print(df_work['inspection_result'].value_counts())

Wrong Pass labels corrected: 70
inspection_result
Pass      10651
Fail       3739
Rework     3610
Name: count, dtype: int64


#### Fill Missing Values with Appropiate Values

In [51]:
# Missing measurements -> median of similar records

for col, group in [
    ("measured_weight_g", "product_id"),
    ("process_temp_c", "product_category"),
    ("inspection_duration_min", "sample_size")
]:
    df_work[col] = (
        df_work[col]
        .fillna(df_work.groupby(group)[col].transform("median"))
        .fillna(df_work[col].median())
        .round(1)
    )

# Text columns that are still blank = unknown
for col in ["supplier_name", "inspector_id", "machine_id", "batch_id",
            "downstream_complaint", "defect_type", "defect_severity", "shift"]:
    df_work[col] = df_work[col].fillna("Unknown")

# Pass results have no root cause
df_work["root_cause_category"] = df_work["root_cause_category"].fillna("Not Applicable")

# Whole numbers should be saved without decimal point (.0)
df_work[["sample_size", "units_defective"]] = df_work[["sample_size", "units_defective"]].astype("Int64")

print(df_work.isna().sum()[df_work.isna().sum() > 0])

inspection_time          216
sample_size                6
units_defective           56
measured_dimension_mm    396
rework_cost_inr           54
scrap_cost_inr            54
dtype: int64


### Final Checks

In [52]:
print(f"Total records: {len(df_work)}")
print(f"Duplicate IDs: {df_work['inspection_id'].duplicated().sum()}")
print(f"Missing dates: {df_work['inspection_date'].isna().sum()}")

print("\nRecords by year:")
print(df_work['inspection_date'].dt.year.value_counts().sort_index())

print(f"\n{df_work['inspection_result'].value_counts().sort_index()}")

Total records: 18000
Duplicate IDs: 0
Missing dates: 0

Records by year:
inspection_date
2025     4472
2026    13528
Name: count, dtype: int64

inspection_result
Fail       3739
Pass      10651
Rework     3610
Name: count, dtype: int64


### Export the Cleaned Dataset

In [53]:
# Save the full cleaned dataset
df_work.to_csv(
    "../data/product_quality_control_cleaned.csv",
    index=False
)

print("Dataset exported successfully!")

Dataset exported successfully!
